# 8.1 FastAPI Local - API REST

API REST simple para búsqueda semántica usando **Qdrant Local** y **Neo4j Desktop**.

## ⚠️ Requisitos:

1. **Qdrant Docker** corriendo
2. **Neo4j Desktop** corriendo
3. **Notebook 7.1** ya ejecutado (función `ask()` disponible)
4. **Puerto 8000** libre

In [1]:
# SOLUCIÓN: Reinstalar todo el stack de ML con versiones compatibles
# El error de disable_datasets_caching indica conflicto de versiones

print("🔄 Reinstalando dependencias con versiones compatibles...")

# Desinstalar versiones problemáticas
%pip uninstall sentence-transformers transformers datasets -y

# Instalar versión estable y compatible de sentence-transformers
%pip install -q sentence-transformers==2.7.0
%pip install -q transformers>=4.21.0,<5.0.0
%pip install -q datasets>=2.0.0

# Otras dependencias
%pip install -q fastapi uvicorn[standard] pydantic qdrant-client neo4j torch numpy nest_asyncio

print("\n✅ Dependencias instaladas")
print("⚠️ IMPORTANTE: Reinicia el kernel después de esto (Kernel > Restart)")

🔄 Reinstalando dependencias con versiones compatibles...
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: datasets 4.3.0
Uninstalling datasets-4.3.0:
  Successfully uninstalled datasets-4.3.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


The system cannot find the file specified.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

✅ Dependencias instaladas
⚠️ IMPORTANTE: Reinicia el kernel después de esto (Kernel > Restart)


In [2]:
# Todas las importaciones necesarias
import os
import numpy as np
import torch
import nest_asyncio

# Verificar versión de transformers antes de importar sentence-transformers
try:
    import transformers
    print(f"✅ transformers versión: {transformers.__version__}")
except ImportError:
    print("⚠️ transformers no instalado correctamente")
    raise

# Verificar sentence-transformers ANTES de cualquier otra importación que lo use
try:
    import sentence_transformers
    print(f"✅ sentence-transformers versión: {sentence_transformers.__version__}")
    
    # Verificar que la función problemática existe o está disponible
    try:
        from sentence_transformers.util import disable_datasets_caching
        print("✅ disable_datasets_caching disponible")
    except ImportError:
        # Si no está disponible, puede estar en otra ubicación (no crítico para uso básico)
        print("ℹ️  disable_datasets_caching no disponible (puede continuar)")
    
except ImportError as e:
    print(f"❌ Error importando sentence-transformers: {e}")
    print("💡 Solución:")
    print("   1. Ejecuta la celda 1 para reinstalar")
    print("   2. Reinicia el kernel (Kernel > Restart)")
    raise

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any, Literal
import uvicorn
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

# Importar SentenceTransformer (el que realmente necesitamos)
try:
    from sentence_transformers import SentenceTransformer
    print("✅ SentenceTransformer importado correctamente")
except ImportError as e:
    print(f"❌ Error importando SentenceTransformer: {e}")
    print("💡 Solución:")
    print("   %pip uninstall sentence-transformers -y")
    print("   %pip install sentence-transformers==2.7.0")
    print("   Luego reinicia el kernel")
    raise

# Habilita ejecutar asyncio en notebooks
nest_asyncio.apply()

print("\n✅ Todas las librerías importadas correctamente")

✅ transformers versión: 4.57.1
✅ sentence-transformers versión: 2.7.0
ℹ️  disable_datasets_caching no disponible (puede continuar)
✅ SentenceTransformer importado correctamente

✅ Todas las librerías importadas correctamente


## Configuración LOCAL


In [3]:
# Configuración LOCAL
os.environ["NEO4J_URI"] = "neo4j://127.0.0.1:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASS"] = "proyectotec"  # Cambiar a tu password

os.environ["QDRANT_URL"] = "http://localhost:6333"
os.environ["QDRANT_API_KEY"] = ""
os.environ["E5_MODEL_NAME"] = "intfloat/multilingual-e5-large-instruct"
os.environ["E5_MAX_LEN"] = "256"

print("✅ Configuración LOCAL lista")
print(f"Neo4j: {os.environ['NEO4J_URI']}")
print(f"Qdrant: {os.environ['QDRANT_URL']}")


✅ Configuración LOCAL lista
Neo4j: neo4j://127.0.0.1:7687
Qdrant: http://localhost:6333


## Clientes e5_query() y ask()


In [4]:
# Clientes lazy-loading
_neo_driver = None
_qdrant_client = None
_e5_encoder = None

def get_neo_driver():
    global _neo_driver
    if _neo_driver is None:
        _neo_driver = GraphDatabase.driver(
            os.environ["NEO4J_URI"],
            auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASS"])
        )
        _neo_driver.verify_connectivity()
    return _neo_driver

def get_qdrant_client():
    global _qdrant_client
    if _qdrant_client is None:
        _qdrant_client = QdrantClient(url=os.environ["QDRANT_URL"], timeout=180)
    return _qdrant_client

def get_encoder():
    global _e5_encoder
    if _e5_encoder is not None:
        return _e5_encoder
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _e5_encoder = SentenceTransformer(os.environ["E5_MODEL_NAME"], device=device)
    _e5_encoder.max_seq_length = int(os.environ["E5_MAX_LEN"])
    return _e5_encoder

@torch.no_grad()
def e5_query(text: str) -> np.ndarray:
    enc = get_encoder()
    vec = enc.encode([f"query: {text}"], normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(vec[0], dtype=np.float32)

print("✅ Clientes inicializados")


✅ Clientes inicializados


In [5]:
# Función ask() - COPIADA DEL NOTEBOOK 7.1
def ask(query_text: str, k: int = 10, type_in: str = "recall"):
    """
    Busca semánticamente en Qdrant y enriquece con Neo4j.
    """
    # 1. Generar embedding
    query_vec = e5_query(query_text)
    
    # 2. Buscar en Qdrant
    collection_name = f"nhtsa_{type_in}s"
    client = get_qdrant_client()
    
    results = client.search(
        collection_name=collection_name,
        query_vector=query_vec.tolist(),
        limit=k,
        with_payload=True
    )
    
    # 3. Extraer IDs del payload
    recalled_ids = []
    hits_data = []
    
    for hit in results:
        recall_id = hit.payload.get('id') or hit.payload.get('camp_no', '')
        if recall_id and recall_id != 'NONE':
            recalled_ids.append(recall_id)
            hits_data.append({
                'id': recall_id,
                'score': hit.score,
                'payload': hit.payload
            })
    
    # 4. Consultar Neo4j
    driver = get_neo_driver()
    enriched = []
    
    with driver.session() as s:
        for data in hits_data:
            recall_id = data['id']
            
            # Complaints no tienen relaciones con Components
            if type_in == 'complaint':
                query = f"""
                MATCH (r:{type_in.capitalize()} {{id: $id}})
                OPTIONAL MATCH (r)-[:OF_MAKE]->(mk:Make)
                OPTIONAL MATCH (r)-[:OF_MODEL]->(md:Model)
                RETURN r, mk.name AS make, md.name AS model, 
                       [r.component] AS components
                """
            else:
                query = f"""
                MATCH (r:{type_in.capitalize()} {{id: $id}})
                OPTIONAL MATCH (r)-[:OF_MAKE]->(mk:Make)
                OPTIONAL MATCH (r)-[:OF_MODEL]->(md:Model)
                OPTIONAL MATCH (r)-[:MENTIONS]->(comp:Component)
                RETURN r, mk.name AS make, md.name AS model, 
                       collect(DISTINCT comp.name) AS components
                """
            result = s.run(query, id=recall_id).single()
            
            if result:
                full_text = data['payload'].get('text', '') or ''
                
                if type_in == 'investigation':
                    subject = result['r'].get('subject', '')
                    summary = result['r'].get('summary', '')
                    full_text = full_text or summary or subject
                    date_field = result['r'].get('open_date', '')
                elif type_in == 'complaint':
                    description = result['r'].get('description', '')
                    full_text = full_text or description
                    date_field = result['r'].get('open_date', '')
                else:
                    subject = result['r'].get('subject', '')
                    full_text = full_text or subject
                    date_field = result['r'].get('recall_date', '')
                
                enriched.append({
                    'id': recall_id,
                    'make': result['make'],
                    'model': result['model'],
                    'year': result['r'].get('year', ''),
                    'component': result['r'].get('component', ''),
                    'components': result['components'],
                    'text': full_text,
                    'date': date_field,
                    'subject': result['r'].get('subject', ''),
                    'summary': result['r'].get('summary', ''),
                    'score': data['score']
                })
    
    return enriched

print("✅ Función ask() lista")


✅ Función ask() lista


## Modelos Pydantic para Request/Response

In [6]:
class SearchRequest(BaseModel):
    question: str = Field(..., description="Pregunta en lenguaje natural", 
                          examples=["airbag sensor failure"])
    type_in: Literal["recall", "investigation", "complaint"] = Field(
        "recall", description="Tipo de entidad a buscar"
    )
    k: int = Field(10, ge=1, le=50, description="Número de resultados")
    
class SearchResponse(BaseModel):
    query: str
    found: int
    results: list

## FastAPI App

In [7]:
app = FastAPI(
    title="NHTSA API",
    description="Búsqueda semántica de Recalls, Investigations y Complaints",
    version="1.0.0"
)

@app.get("/health")
def health():
    return {"ok": True, "message": "API funcionando correctamente"}

@app.post("/search", response_model=SearchResponse)
def search_endpoint(req: SearchRequest):
    """
    Endpoint principal de búsqueda semántica.
    
    Args:
        req: SearchRequest con question, type_in, k
    
    Returns:
        Lista de resultados enriquecidos con metadatos de Neo4j
    """
    try:
        results = ask(req.question, k=req.k, type_in=req.type_in)
        return SearchResponse(
            query=req.question,
            found=len(results),
            results=results
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## Iniciar Servidor

In [8]:
import threading
import time

# Configuración del servidor
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# Función para ejecutar el servidor en un hilo
def run_server():
    nest_asyncio.apply()
    server.run()

# Iniciar servidor en hilo separado
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("="*70)
print("INICIANDO SERVIDOR FASTAPI")
print("="*70)
print("\n📍 Local:  http://127.0.0.1:8000")
print("📍 Docs:   http://127.0.0.1:8000/docs")
print("📍 Health: http://127.0.0.1:8000/health")
print("\n" + "="*70)
print("⚠️  Servidor corriendo en segundo plano")
print("⚠️  Para detener: Kernel > Restart")
print("="*70 + "\n")

# Esperar un poco para que el servidor inicie
time.sleep(2)

INICIANDO SERVIDOR FASTAPI

📍 Local:  http://127.0.0.1:8000
📍 Docs:   http://127.0.0.1:8000/docs
📍 Health: http://127.0.0.1:8000/health

⚠️  Servidor corriendo en segundo plano
⚠️  Para detener: Kernel > Restart



INFO:     Started server process [21156]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
